In [16]:
import sqlite3
import json
import time
import os
import re
from pathlib import Path

In [17]:
WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True)

In [18]:
# z zadania 1

import functools
from datasets import load_dataset

@functools.lru_cache(maxsize=4)
def get_imdb_subset(split: str, n: int):
    ds = load_dataset("stanfordnlp/imdb", split=split).shuffle(seed=42).select(range(n))
    return [(r["text"], r["label"]) for r in ds]

samples = get_imdb_subset("train", 2000)

In [19]:
#baza SQL

DB_PATH = WORKDIR / "imdb.db"

if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(str(DB_PATH))
cur = conn.cursor()

cur.execute("""
CREATE TABLE reviews (
    id INTEGER PRIMARY KEY,
    text TEXT NOT NULL,
    label INTEGER NOT NULL,
    word_count INTEGER,
    char_count INTEGER
)
""")

start_insert_sql = time.time()

for i, (text, label) in enumerate(samples):
    cur.execute(
        """
        INSERT INTO reviews (id, text, label, word_count, char_count)
        VALUES (?, ?, ?, ?, ?)
        """,
        (i, text, label, len(text.split()), len(text))
    )

conn.commit()
insert_sql_time = time.time() - start_insert_sql

print("Baza SQL została utworzona.")
print(f"Czas wstawiania SQL: {insert_sql_time:.4f}s")

sql_queries = {
    "Rozkład klas + średni word_count": """
        SELECT label, COUNT(*) AS liczba, AVG(word_count) AS avg_word_count
        FROM reviews
        GROUP BY label
    """,
    "Zakres długości": """
        SELECT MIN(word_count), MAX(word_count)
        FROM reviews
    """,
    "Recenzje powyżej 500 słów": """
        SELECT COUNT(*)
        FROM reviews
        WHERE word_count > 500
    """
}

start_read_sql = time.time()

print("Zapytania SQL:")

for name, query in sql_queries.items():
    print(f"\n{name}")
    for row in cur.execute(query):
        print(row)

read_sql_time = time.time() - start_read_sql

conn.close()

Baza SQL została utworzona.
Czas wstawiania SQL: 0.0787s
Zapytania SQL:

Rozkład klas + średni word_count
(0, 1000, 224.705)
(1, 1000, 232.164)

Zakres długości
(12, 1005)

Recenzje powyżej 500 słów
(164,)


In [20]:
# NoSQL-style

DB_JSON = WORKDIR / "imdb_json.db"

if DB_JSON.exists():
    DB_JSON.unlink()

conn2 = sqlite3.connect(str(DB_JSON))
cur2 = conn2.cursor()

cur2.execute("""
CREATE TABLE reviews_json (
    id INTEGER PRIMARY KEY,
    doc TEXT NOT NULL
)
""")

start_insert_json = time.time()

for i, (text, label) in enumerate(samples):
    words = re.findall(r"\w+", text.lower())

    tags = []
    for word in words:
        if len(word) > 5 and word not in tags:
            tags.append(word)
        if len(tags) == 3:
            break

    doc = {
        "text": text,
        "label": label,
        "stats": {
            "word_count": len(text.split()),
            "char_count": len(text),
            "sentiment_hint": "pos" if label == 1 else "neg"
        },
        "tags": tags
    }

    cur2.execute(
        "INSERT INTO reviews_json (id, doc) VALUES (?, ?)",
        (i, json.dumps(doc, ensure_ascii=False))
    )

conn2.commit()
insert_json_time = time.time() - start_insert_json

print("Baza JSON została utworzona.")
print(f"Czas wstawiania JSON: {insert_json_time:.4f}s")


queries = {
    "Rozkład klas": """
        SELECT
            json_extract(doc, '$.stats.sentiment_hint') AS hint,
            COUNT(*) AS liczba
        FROM reviews_json
        GROUP BY hint
    """,

    "Średni word_count dla każdej klasy": """
        SELECT
            json_extract(doc, '$.stats.sentiment_hint') AS hint,
            AVG(json_extract(doc, '$.stats.word_count')) AS avg_word_count
        FROM reviews_json
        GROUP BY hint
    """,

    "Recenzje, gdzie tags zawiera movie": """
        SELECT
            id,
            json_extract(doc, '$.stats.sentiment_hint') AS hint,
            json_extract(doc, '$.tags') AS tags
        FROM reviews_json
        WHERE json_extract(doc, '$.tags') LIKE '%movie%'
        LIMIT 10
    """,

    "Top 5 najdłuższych pozytywnych recenzji": """
        SELECT
            id,
            json_extract(doc, '$.stats.word_count') AS word_count,
            substr(json_extract(doc, '$.text'), 1, 100) AS fragment
        FROM reviews_json
        WHERE json_extract(doc, '$.label') = 1
        ORDER BY json_extract(doc, '$.stats.word_count') DESC
        LIMIT 5
    """
}

start_read_json = time.time()

print("Zapytania JSON / NoSQL-style:")

for name, query in queries.items():
    print(f"\n{name}")
    for row in cur2.execute(query):
        print(row)

read_json_time = time.time() - start_read_json

conn2.close()

Baza JSON została utworzona.
Czas wstawiania JSON: 0.2709s
Zapytania JSON / NoSQL-style:

Rozkład klas
('neg', 1000)
('pos', 1000)

Średni word_count dla każdej klasy
('neg', 224.705)
('pos', 232.164)

Recenzje, gdzie tags zawiera movie
(50, 'pos', '["totally","movies","journey"]')
(61, 'neg', '["movies","before","sittings"]')
(64, 'pos', '["gangster","movies","should"]')
(122, 'neg', '["movies","premiere","extremely"]')
(137, 'neg', '["considered","myself","movies"]')
(147, 'neg', '["possibly","movies","critics"]')
(150, 'neg', '["movies","should","seemed"]')
(193, 'pos', '["famous","movies","french"]')
(268, 'pos', '["favourite","movies","dialogue"]')
(281, 'neg', '["please","without","movies"]')

Top 5 najdłuższych pozytywnych recenzji
(1109, 982, 'The interesting aspect of "The Apprentice" is it demonstrates that the traditional job interview and')
(1557, 973, 'Yesterday, I went to the monthly Antique Flea Market that comes to town. I really have no interest i')
(1526, 969, '******

In [21]:
# POROWNANIE

size_sql = os.path.getsize(DB_PATH)
size_json = os.path.getsize(DB_JSON)

print("Porównanie:")
print(f"SQL schema (reviews):       {size_sql:>9,} bajtów")
print(f"JSON schema (reviews_json): {size_json:>9,} bajtów")

print(f"\nCzas wstawiania SQL:  {insert_sql_time:.4f}s")
print(f"Czas wstawiania JSON: {insert_json_time:.4f}s")

print(f"\nCzas czytania SQL:  {read_sql_time:.4f}s")
print(f"Czas czytania JSON: {read_json_time:.4f}s")

Porównanie:
SQL schema (reviews):       3,215,360 bajtów
JSON schema (reviews_json): 3,600,384 bajtów

Czas wstawiania SQL:  0.0787s
Czas wstawiania JSON: 0.2709s

Czas czytania SQL:  0.0064s
Czas czytania JSON: 0.0375s


### Wnioski

Oba schematy przechowywały te same dane i pozwoliły uzyskać identyczne wyniki analityczne, jednak klasyczny model relacyjny okazał się bardziej wydajny.

Rozmiar bazy SQL wyniósł 3 215 360 bajtów, natomiast baza wykorzystująca dokumenty JSON zajmowała 3 600 384 bajty. Oznacza to, że przechowywanie danych w postaci JSON wymagało około 12% więcej miejsca na dysku ze względu na powtarzające się nazwy pól i dodatkową strukturę dokumentów.

Różnice były jeszcze bardziej widoczne pod względem wydajności. Wstawienie danych do tabeli SQL zajęło 0.0684 s, podczas gdy zapis dokumentów JSON wymagał 0.2747 s, czyli około czterokrotnie więcej czasu. Podobnie było podczas odczytu danych: zapytania SQL wykonały się w 0.0057 s, a analogiczne zapytania wykorzystujące `json_extract()` potrzebowały 0.0395 s, czyli prawie siedmiokrotnie dłużej.

Dla tego problemu lepszym wyborem jest klasyczny model SQL. Dane mają stałą strukturę, często wykonujemy agregacje oraz filtrowanie po konkretnych polach, dlatego przechowywanie ich w osobnych kolumnach jest bardziej efektywne zarówno pod względem szybkości działania, jak i zajmowanego miejsca. Schemat oparty na JSON byłby korzystniejszy w sytuacji, gdy dokumenty miałyby różną strukturę lub często zmieniający się zestaw pól.